# Python Environments — Telemetry Dashboard

Interactive notebook for running the Kusto queries in `analysis/kusto/*.kql`.

**Prerequisites:**
1. `pip install -r requirements.txt`
2. `az login` (Azure CLI authentication)
3. `nbstripout --install` (one-time setup to auto-strip notebook outputs before commits)

In [ ]:
from initialize import initialize
from query_runner import run_kql, run_kql_file, load_kql_sections
from IPython.display import display, Markdown, HTML
import pandas as pd
import altair as alt

# Show all rows and wrap long column text
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


client = initialize()
print("Connected to Kusto.")

---
## 0. Telemetry Validation

Checks that all telemetry events are arriving with correct properties.

In [ ]:
sections = load_kql_sections("00-telemetry-validation.kql")
for title, query in sections[:4]:
    display(Markdown(f"### {title}"))
    try:
        df = run_kql(client, query)
        display(df)
        if "TotalUniqueMachines" in df.columns:
            print(f"Total unique machines: {df['TotalUniqueMachines'].iloc[0]:,}")
    except Exception as e:
        print(f"  ⚠️ {e}")

In [ ]:
for title, query in sections[4:]:
    display(Markdown(f"### {title}"))
    try:
        df = run_kql(client, query)
        display(df)
        if "TotalUniqueMachines" in df.columns:
            print(f"Total unique machines: {df['TotalUniqueMachines'].iloc[0]:,}")
    except Exception as e:
        print(f"  ⚠️ {e}")

---
## 1. Registration Failure Stage Breakdown (28 days)

For each manager that failed, shows **which stage** in the registration flow broke.
`failureStage` is hierarchical (e.g. `getPipenv:nativeFinderRefresh`). High counts at a specific stage = priority fix target.

In [ ]:
df = run_kql_file(client, "01-failure-stage-breakdown.kql")
display(df)

---
## 2. Error Type × Failure Stage Matrix (28 days)

Cross-tabulates **error type** (what kind of error) with **failure stage** (where it happened).
This is the key diagnostic view — e.g. `connection_error` at `nativeFinderRefresh` means PET died during native finder.

In [ ]:
df = run_kql_file(client, "02-error-type-x-failure-stage.kql")
display(df)

---
## 3. Daily Registration Failures by Manager & Stage (14 days)

Day-by-day failure counts per manager + stage. A spike in one stage on a specific day = regression in that code path.

In [ ]:
df = run_kql_file(client, "03-failure-stage-daily-trend.kql")
display(df)

---
## 4. Overall Setup Success Rate (28 days)

Top-level health metric. If this drops, something is broken.

In [ ]:
df = run_kql_file(client, "04-overall-setup-success-rate.kql")
display(df)

---
## 5. Manager Availability

What tools do users actually have installed? Shows registered vs skipped vs failed per manager.

In [ ]:
df = run_kql_file(client, "05-manager-availability.kql")
display(df)

if not df.empty:
    melted = df.melt(
        id_vars="Manager",
        value_vars=["Registered", "Skipped", "Failed"],
        var_name="Status",
        value_name="Machines",
    )
    chart = (
        alt.Chart(melted)
        .mark_bar()
        .encode(
            x=alt.X("Manager:N", sort="-y"),
            y=alt.Y("Machines:Q"),
            color=alt.Color("Status:N", scale=alt.Scale(
                domain=["Registered", "Skipped", "Failed"],
                range=["#4c78a8", "#e45756", "#f58518"],
            )),
            tooltip=["Manager", "Status", "Machines"],
        )
        .properties(width=600, height=300, title="Manager Availability")
    )
    display(chart)

---
## 6. Daily Trend (14 days)

Day-by-day trend of setup success rate. Check after shipping a new version.

In [ ]:
df = run_kql_file(client, "06-daily-trend.kql")
display(df)

if not df.empty:
    chart = (
        alt.Chart(df)
        .mark_line(point=True)
        .encode(
            x=alt.X("Day:T", title="Date"),
            y=alt.Y("SuccessRate:Q", title="Success Rate (%)", scale=alt.Scale(domain=[0, 100])),
            tooltip=["Day:T", "SuccessRate:Q", "SuccessCount:Q", "ErrorCount:Q"],
        )
        .properties(width=700, height=300, title="Daily Setup Success Rate")
    )
    display(chart)

---
## 7. Error Type Distribution

Groups all failures by error type across setup and individual managers.

In [ ]:
df = run_kql_file(client, "07-error-type-distribution.kql")
display(df)

if not df.empty:
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X("ErrorType:N", sort="-y", title="Error Type"),
            y=alt.Y("AffectedMachines:Q", title="Affected Machines"),
            color="Source:N",
            tooltip=["ErrorType", "Source", "EventCount", "AffectedMachines"],
        )
        .properties(width=600, height=300, title="Error Type Distribution")
    )
    display(chart)

---
## 8. Hang ↔ Failure Correlation

Do hangs always cause failures, or do some self-recover?

In [ ]:
df = run_kql_file(client, "08-hang-failure-correlation.kql")
display(df)

---
## 9. Weekly Health Summary

One-stop query for weekly check. Returns all key numbers in a single row.

In [ ]:
df = run_kql_file(client, "09-weekly-health-summary.kql")
display(df)